## Local Agent with LMStudio and Cogito v1

In this example we will learn how to use fully local LLMs, implementing the new cogito v1 models (specifically `cogito-v1-preview-qwen-32b`). These models are incredibly competent LLMs fully competent of powering agentic workflows with tool calling, while being small enough to run on personal hardware.

We will be using LM Studio to host Cogito models locally, installation instructions can be found [here](https://lmstudio.ai/download).

LM Studio (on Mac) supports all GGUF quantized models. That means that we must use [`lmstudio-community/cogito-v1-preview-qwen-32b`](https://huggingface.co/lmstudio-community/cogito-v1-preview-qwen-32b) which can be downloaded for LM Studio [here](https://model.lmstudio.ai/download/lmstudio-community/cogito-v1-preview-qwen-32b).

## Using Cogito v1

Once the model has been downloaded we can select **Start server on port 1234** in our LM Studio interface (accessible by clicking the icon in the taskbar) and load our chosen model. Then we confirm LM Studio is accessible like so:

In [1]:
!curl http://localhost:1234/v1/models

{
  "data": [
    {
      "id": "cogito-v1-preview-qwen-32b",
      "object": "model",
      "owned_by": "organization_owner"
    },
    {
      "id": "cogito-v1-preview-llama-70b",
      "object": "model",
      "owned_by": "organization_owner"
    },
    {
      "id": "unsloth/llama-4-scout-17b-16e-instruct",
      "object": "model",
      "owned_by": "organization_owner"
    },
    {
      "id": "lmstudio-community/llama-4-scout-17b-16e-instruct",
      "object": "model",
      "owned_by": "organization_owner"
    },
    {
      "id": "text-embedding-nomic-embed-text-v1.5",
      "object": "model",
      "owned_by": "organization_owner"
    },
    {
      "id": "mistral-small-3.1-24b-instruct-2503",
      "object": "model",
      "owned_by": "organization_owner"
    }
  ],
  "object": "list"
}

We can take the model name from above and insert it into our `model` parameter below (after `lm_studio/`):

In [2]:
from litellm import completion
import os

MODEL = "lm_studio/cogito-v1-preview-qwen-32b"

# set to the port LM studio is using, default is 1234
os.environ["LM_STUDIO_API_BASE"] = "http://localhost:1234/v1"

response = completion(
    model=MODEL,
    messages=[{"role": "user", "content": "Hello, how are you?"}],
    api_key="sk-some-api-key"  # need a dummy API key
)
print(response)

ModelResponse(id='chatcmpl-4xj4ueewz0im6a40v6ykm9', created=1744611855, model='lm_studio/cogito-v1-preview-qwen-32b', object='chat.completion', system_fingerprint='cogito-v1-preview-qwen-32b', choices=[Choices(finish_reason='stop', index=0, message=Message(content="I'm doing well, thank you! How can I help you today?", role='assistant', tool_calls=None, function_call=None, provider_specific_fields={'refusal': None, 'annotations': None}))], usage=Usage(completion_tokens=15, prompt_tokens=14, total_tokens=29, completion_tokens_details=None, prompt_tokens_details=None), service_tier=None, stats={})


## Async Streaming

Let's see how we can implement asynchronous streaming via LiteLLM.

In [3]:
from litellm import acompletion

response = await acompletion(
    model=MODEL,
    messages=[{"role": "user", "content": "Hello, how are you?"}],
    api_key="sk-some-api-key",
    stream=True
)

async for chunk in response:
    print(chunk)

ModelResponseStream(id='chatcmpl-7acf4b8d-84f5-4185-97f4-36396275cb4d', created=1744611885, model='cogito-v1-preview-qwen-32b', object='chat.completion.chunk', system_fingerprint='cogito-v1-preview-qwen-32b', choices=[StreamingChoices(finish_reason=None, index=0, delta=Delta(provider_specific_fields=None, refusal=None, content='Hi', role='assistant', function_call=None, tool_calls=None, audio=None), logprobs=None)], provider_specific_fields=None, stream_options=None, citations=None)
ModelResponseStream(id='chatcmpl-7acf4b8d-84f5-4185-97f4-36396275cb4d', created=1744611885, model='cogito-v1-preview-qwen-32b', object='chat.completion.chunk', system_fingerprint='cogito-v1-preview-qwen-32b', choices=[StreamingChoices(finish_reason=None, index=0, delta=Delta(provider_specific_fields=None, refusal=None, content='!', role=None, function_call=None, tool_calls=None, audio=None), logprobs=None)], provider_specific_fields=None, stream_options=None, citations=None)
ModelResponseStream(id='chatcmpl

We can parse out the output here like so:

In [4]:
response = await acompletion(
    model=MODEL,
    messages=[{"role": "user", "content": "Hello, how are you?"}],
    api_key="sk-some-api-key",
    stream=True
)

async for chunk in response:
    if (token := chunk.choices[0].delta.content) is not None:
        print(token, end="", flush=True)

I'm doing well, thank you! How can I help you today?

## Tool Calls

Tool calling is naturally a big part of what makes agents useful — let's see how we can do this with Mistral small. First, we would usually check if our model can use function calling via `supports_function_calling`:

In [5]:
from litellm import supports_function_calling

supports_function_calling(MODEL)

False

This tells us we _cannot_ use function calling, but this is not entirely true. We _can_ use function calling but we must use LM Studio's OpenAI chat completion endpoint. This endpoint _does not_ use OpenAI, but simply replicates the pattern of OpenAI's chat completion endpoint. The method for calling this is slightly different, synchronously we do it like so:

In [6]:
OAI_MODEL = MODEL.replace("lm_studio/", "openai/")  # swap lm_studio/ for openai/

response = completion(
    model=OAI_MODEL,
    messages=[{"role": "user", "content": "Hello, how are you?"}],
    api_key="sk-some-api-key",
    base_url="http://localhost:1234/v1",  # and modify the base_url
)

response

ModelResponse(id='chatcmpl-yl38334za7o1vwzj023f', created=1744612039, model='cogito-v1-preview-qwen-32b', object='chat.completion', system_fingerprint='cogito-v1-preview-qwen-32b', choices=[Choices(finish_reason='stop', index=0, message=Message(content="I'm doing well, thank you for asking! How can I help you today?", role='assistant', tool_calls=None, function_call=None, provider_specific_fields={'refusal': None, 'annotations': None}))], usage=Usage(completion_tokens=17, prompt_tokens=14, total_tokens=31, completion_tokens_details=None, prompt_tokens_details=None), service_tier=None, stats={})

All we changed here is:

* We override the default `openai/` URL via `base_url`
* We swap `lm_studio/` for `openai/` in the `model` name.

The pattern is the same for async streaming:

In [7]:
response = await acompletion(
    model=OAI_MODEL,
    messages=[{"role": "user", "content": "Hello, how are you?"}],
    stream=True,
    api_key="sk-some-api-key",
    base_url="http://localhost:1234/v1",
)

async for chunk in response:
    if (token := chunk.choices[0].delta.content) is not None:
        print(token, end="", flush=True)

I'm doing well, thank you! How can I help you today?

Now we have that, let's begin by defining a few tools. The first of those will enable access to the web for our agent via the SerpAPI — for which you can get a free API key [here](https://serpapi.com/dashboard):

In [8]:
from getpass import getpass
import aiohttp

SERPAPI_API_KEY = getpass("Enter your SerpAPI API key: ")

# define our search parameters
params = {
    "api_key": SERPAPI_API_KEY,
    "engine": "google",
    "q": "latest world news"
}

async with aiohttp.ClientSession() as session:
    async with session.get(
        "https://serpapi.com/search",
        params=params
    ) as response:
        results = await response.json()

results["organic_results"]

[{'position': 1,
  'title': 'World | Latest News & Updates',
  'link': 'https://www.bbc.com/news/world',
  'redirect_link': 'https://www.google.com/url?sa=t&source=web&rct=j&opi=89978449&url=https://www.bbc.com/news/world&ved=2ahUKEwjcpO726NaMAxWCkYkEHV-GA-sQFnoECCoQAQ',
  'displayed_link': 'https://www.bbc.com › news › world',
  'favicon': 'https://serpapi.com/searches/67fca0f1477ecfd134bcb3af/images/16c8f2982b167f589c032bebd52ffcaac23c298b281c232418cab8d216c47a55.png',
  'date': '3 hours ago',
  'snippet': "World · Trump says no one 'off the hook' as he suggests new Chinese tariffs · Ukraine's allies condemn Russia over deadly missile attack · Israeli air strike ...",
  'snippet_highlighted_words': ["Trump says no one 'off the hook' as he suggests new Chinese tariffs"],
  'sitelinks': {'inline': [{'title': 'BBC World News',
     'link': 'https://www.bbc.com/news/world_radio_and_tv'},
    {'title': 'Africa', 'link': 'https://www.bbc.com/news/world/africa'},
    {'title': 'Europe', 'li

This is our _async_ call to perform Google searches via SerpAPI, the results are pretty heavy so we can organize them with pydantic like so:

In [9]:
from pydantic import BaseModel

class Article(BaseModel):
    title: str
    source: str
    link: str
    snippet: str

    @classmethod
    def from_serpapi_result(cls, result: dict) -> "Article":
        return cls(
            title=result["title"],
            source=result["source"],
            link=result["link"],
            snippet=result["snippet"],
        )
    
    def __str__(self) -> str:
        return f"## {self.title} - ({self.source})\n_{self.link}_\n{self.snippet}\n"
    
articles = [Article.from_serpapi_result(result) for result in results["organic_results"]]
articles

[Article(title='World | Latest News & Updates', source='BBC', link='https://www.bbc.com/news/world', snippet="World · Trump says no one 'off the hook' as he suggests new Chinese tariffs · Ukraine's allies condemn Russia over deadly missile attack · Israeli air strike ..."),
 Article(title='World news - breaking news, video, headlines and opinion', source='CNN', link='https://www.cnn.com/world', snippet='View CNN world news today for international news and videos from Europe, Asia, Africa, the Middle East and the Americas.'),
 Article(title='World News | Latest Top Stories', source='Reuters', link='https://www.reuters.com/world/', snippet='World · Former South Korean mayor of Daegu city announces bid for president · Indonesia arrests judge after palm oil companies cleared of graft charges.'),
 Article(title='Latest news from around the world', source='The Guardian', link='https://www.theguardian.com/world', snippet="Latest World news news, comment and analysis from the Guardian, the wor

We format all of this into a single function that our LLM will be able to call:

In [12]:
async def web_search(query: str) -> list[Article]:
    """Use this function to search the web for information. Provide natural language to the
    query with as much context as possible to get the best results.
    """
    params = {
        "api_key": SERPAPI_API_KEY,
        "engine": "google",
        "q": query
    }
    
    async with aiohttp.ClientSession() as session:
        async with session.get(
            "https://serpapi.com/search",
            params=params
        ) as response:
            results = await response.json()
            
    articles = [Article.from_serpapi_result(result) for result in results["organic_results"]]
    articles = "\n".join([str(article) for article in articles])
    return articles

Then we parse these tools into a list of function schemas that our LLM will be able to read:

In [13]:
from graphai.utils import get_schemas

tools = get_schemas(callables=[web_search], format="default")
tools[0]["function"]["parameters"]["properties"]["query"]["description"] = "The query to search the web for"
tools

[{'type': 'function',
  'function': {'name': 'web_search',
   'description': 'Use this function to search the web for information. Provide natural language to the\nquery with as much context as possible to get the best results.',
   'parameters': {'type': 'object',
    'properties': {'query': {'description': 'The query to search the web for',
      'type': 'string'}},
    'required': ['query']}}}]

In [15]:
query = {"role": "user", "content": "tell me about the latest world news"}

response = completion(
    model=OAI_MODEL,
    messages=[query],
    tools=tools,
    tool_choice="auto",
    api_key="sk-some-api-key",
    base_url="http://localhost:1234/v1",
)

(
    response.choices[0].message.tool_calls[0].function.name,
    response.choices[0].message.tool_calls[0].function.arguments,
)

('web_search', '{"query":"latest world news"}')

Our LLM has generated the tool choice and input parameters for our tool but we have not executed the tool, we must handle that ourselves. To do so we will create a mapping from tool names to their functions.

In [16]:
tool_map = {
    "web_search": web_search
    # when using multiple tools, we would add them here
}

Now we execute the tool like so:

In [17]:
from IPython.display import Markdown, display

tool_out = await tool_map[response.choices[0].message.tool_calls[0].function.name](
    response.choices[0].message.tool_calls[0].function.arguments
)
display(Markdown(tool_out))

## World News - Latest and Breaking Coverage - (Yahoo News)
_https://news.yahoo.com/world/_
The latest world news and headlines from Yahoo News and international ... Search query. Advertisement. World. Sports·Yahoo Sports. Masters 2025 payouts ...

## Top & Breaking World News Today - (AP News)
_https://apnews.com/world-news_
Religion · Español · Standards · Quizzes · Press Releases · My Account · Sign in. Search Query Submit Search. Show Search Menu. Submit Search. World · Israel- ...

## World News - Latest and Breaking Coverage - (Yahoo)
_https://www.yahoo.com/news/world/_
The latest world news and headlines from Yahoo News and international ... Search query. Advertisement. World. Sports·Yahoo Sports. Masters 2025: Augusta ...

## Latest World News Today | International News Headlines - (Mint)
_https://www.livemint.com/news/world_
Query, Suggestion. Your Message. footLogo. Connect with us: footLogo. trending stories. Board Exam Results 2025 AP ...

## Page 2 - World News - (Mint)
_https://www.livemint.com/news/world/page-2_
Query, Suggestion. Your Message. footLogo. Connect with us: footLogo. trending stories. Weather today Gold prices today Mahavir jayanti ...

## World News - (Wion)
_https://www.wionews.com/world?page=1561_
Get top and latest World News - Read Breaking World News and World News Headlines ... query: 'Am I speaking English to you?' By Prapti Upadhayay. Jul 11, 2024 21 ...

## World News Index - (worldnewsindex.com)
_https://worldnewsindex.com/world-news/_
Type your search query and hit enter: World · Europe · U.S. · U.K. · France · Business ... The latest World News from the World News Index.

## Latest World News, International News | Breaking World News - (The Express Tribune)
_https://tribune.com.pk/WORLD/archives?page=1049_
Find the latest world news and International news headlines today ... query. it took indian army 15 months to prepare for cross loc surgical strike ...

## News: Today's News Headlines, Breaking News India, World ... - (Hindustan Times)
_https://www.hindustantimes.com/_
Axar's savage response to Karthik's 'disappointing' query after MI beat DC. After the match, commentator Murali Karthik asked DC skipper Axar Patel about his ...

## Read Latest News Live, India's Latest News Today - (Hindustan Times)
_https://www.hindustantimes.com/latest-news_
Axar's savage response to Karthik's 'disappointing' query after MI beat DC. After the match, commentator Murali Karthik asked DC skipper Axar Patel about his ...


We then format this and the initial tool call from our LLM into messages, and feed them back into our LLM for a final response.

In [18]:
tool_call = {"role": "assistant", "content": response.choices[0].message.content, "tool_calls": response.choices[0].message.tool_calls, "tool_call_id": response.choices[0].message.tool_calls[0].id}
tool_exec = {"role": "tool", "content": tool_out, "tool_call_id": response.choices[0].message.tool_calls[0].id}

In [19]:
messages = [query, tool_call, tool_exec]

response = completion(
    model=OAI_MODEL,
    messages=messages,
    tool_choice="auto",
    api_key="sk-some-api-key",
    base_url="http://localhost:1234/v1",
    tools=tools
)
response.choices[0]

Choices(finish_reason='stop', index=0, message=Message(content='Here are some of the latest world news headlines from various sources:\n\n1. **Masters 2025 Payouts**: Yahoo News is covering upcoming Masters golf tournament payouts for 2025.\n   \n2. **AP News**: Breaking coverage on international events including developments in Israel and other global issues.\n\n3. **World News - Yahoo**: Coverage of the latest news from around the world, with highlights on sports and Augusta\'s Masters event.\n\n4. **Latest World News (Mint)**: International headlines featuring trends like board exam results and current affairs.\n\n5. **WION**: Breaking news articles covering various global issues and statements, including a notable article titled "\'Am I speaking English to you?\'" from July 11, 2024.\n\n6. **World News Index**: Aggregated top world news stories categorized by regions like Europe, U.S., UK, France, etc.\n\n7. **The Express Tribune**: Breaking news focusing on international developme

That looks good! We can wrap all of this up into some easier to use agentic logic to keep track of the conversation, execute tools when needed, etc, like so:

In [20]:
import json
from typing import Callable


class Agent:
    def __init__(self, tools: list[Callable]):
        self.tools = tools
        self.function_schemas = get_schemas(tools)
        self.mapping = {tool.__name__: tool for tool in tools}
        self.messages = [
            {
                "role": "system",
                "content": (
                    "You are a helpful assistant that can use tools to help answer questions "
                    "for the user."
                )
            },
        ]
        
    async def __call__(self, query: str, max_iterations: int = 3) -> str:
        self.messages.append({"role": "user", "content": query})
        i = 0
        while i < max_iterations:
            response = await acompletion(
                model=OAI_MODEL,
                messages=self.messages,
                tools=self.function_schemas,
                tool_choice="auto",
                api_key="sk-some-api-key",
                base_url="http://localhost:1234/v1",
            )
            # check if we got a tool call
            if (tool_calls := response.choices[0].message.tool_calls):
                tool_name = tool_calls[0].function.name
                tool_args = json.loads(tool_calls[0].function.arguments)
                tool_call_id = tool_calls[0].id
            else:
                tool_calls = None
                tool_name = None
                tool_args = None
                tool_call_id = None
            # append assistant message
            self.messages.append({
                "role": "assistant",
                "content": response.choices[0].message.content,
                "tool_calls": tool_calls,
                "tool_call_id": tool_call_id
            })
            if tool_calls:
                # if we got a tool call, we execute it and add the message to the conversation
                tool_out = await self.mapping[tool_name](**tool_args)
                self.messages.append({
                    "role": "tool", "content": tool_out, "tool_call_id": tool_call_id
                })
                i += 1
            else:
                # if we didn't get a tool call we assume the iteration is complete
                break
        return self.messages[-1]

In [21]:
agent = Agent(tools=[web_search])
out = await agent("tell me about the latest world news")
out

{'role': 'assistant',
 'content': 'Here are some of the latest major headlines from around the world:\n\n1. **Trump Tariffs**: Donald Trump has suggested imposing new tariffs on China and stated that no one will be "off the hook". This comes amid ongoing tensions between the two countries.\n\n2. **Ukraine Conflict**: Ukraine\'s European allies have condemned Russia following a deadly missile attack. The situation in Ukraine continues to dominate international news.\n\n3. **Middle East**:\n   - Israel has carried out an airstrike that destroyed what appears to be Gaza City\'s last functioning hospital, leading to international condemnation.\n   - Palestinian rescuers are working at a site where an Israeli strike killed multiple civilians.\n\n4. **Australia Election**: The upcoming Australian election is making headlines with key debates between Prime Minister Albanese and opposition leader Dutton scheduled for the days before polling day.\n\n5. **South Africa**: There has been news of a

In [22]:
display(Markdown(out["content"]))

Here are some of the latest major headlines from around the world:

1. **Trump Tariffs**: Donald Trump has suggested imposing new tariffs on China and stated that no one will be "off the hook". This comes amid ongoing tensions between the two countries.

2. **Ukraine Conflict**: Ukraine's European allies have condemned Russia following a deadly missile attack. The situation in Ukraine continues to dominate international news.

3. **Middle East**:
   - Israel has carried out an airstrike that destroyed what appears to be Gaza City's last functioning hospital, leading to international condemnation.
   - Palestinian rescuers are working at a site where an Israeli strike killed multiple civilians.

4. **Australia Election**: The upcoming Australian election is making headlines with key debates between Prime Minister Albanese and opposition leader Dutton scheduled for the days before polling day.

5. **South Africa**: There has been news of an American pastor being kidnapped in South Africa, sparking international concern.

6. **Indonesia**: A judge has been arrested following a controversial decision where palm oil companies were cleared of corruption charges.

7. **South Korea**: The former mayor of Daegu city has announced his candidacy for the upcoming presidential election.

These stories are from various major news outlets including BBC, CNN, Reuters, and others. For more detailed coverage on any specific story, I'd recommend visiting these news sources directly.

---